<a href="https://colab.research.google.com/github/Deelaksha22/AI-model/blob/main/LLM_Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install -q -U langgraph langchain-groq

import os
from typing import TypedDict
from google.colab import userdata
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

class State(TypedDict):
    question: str
    category: str
    answer: str

def classify(state: State):
    response = llm.invoke(
        f"""Classify this question into exactly one category:
technical, general, or other.

Question: {state["question"]}

Return only the category name."""
    )

    return {
        "category": response.content.strip().lower()
    }

def answer(state: State):
    response = llm.invoke(
        f"""Answer the following question clearly and accurately:

{state["question"]}"""
    )

    return {
        "answer": response.content
    }

def fallback(state: State):
    return {
        "answer": "Sorry, I can only answer technical and general questions."
    }

def route(state: State):
    if state["category"] in ["technical", "general"]:
        return "answer"
    return "fallback"

graph = StateGraph(State)

graph.add_node("classify", classify)
graph.add_node("answer", answer)
graph.add_node("fallback", fallback)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    route,
    {
        "answer": "answer",
        "fallback": "fallback"
    }
)

graph.add_edge("answer", END)
graph.add_edge("fallback", END)

app = graph.compile()

question = input("Enter your question: ")

result = app.invoke({
    "question": question,
    "category": "",
    "answer": ""
})

print("\nCategory:", result["category"])
print("\nAnswer:", result["answer"])

Enter your question: code to print hello world in java

Category: technical

Answer: Here’s the classic “Hello, World!” program in Java:

```java
public class HelloWorld {
    public static void main(String[] args) {
        System.out.println("Hello, World!");
    }
}
```

**How to run it**

1. **Save the file**  
   Save the code above in a file named `HelloWorld.java`. The file name must match the public class name (`HelloWorld`).

2. **Compile**  
   Open a terminal/command prompt, navigate to the directory containing the file, and compile it with the Java compiler:

   ```bash
   javac HelloWorld.java
   ```

   This creates a `HelloWorld.class` byte‑code file.

3. **Execute**  
   Run the compiled program:

   ```bash
   java HelloWorld
   ```

   You should see the output:

   ```
   Hello, World!
   ```

That’s all there is to it—just a single class with a `main` method that prints the string to the standard output.
